# ⚽ Predict the FIFA World Cup 2026

## 📖 Background

The 2026 FIFA World Cup is one of the biggest sporting events in the world, hosted across the United States, Canada, and Mexico. For the first time, the tournament expands to 48 teams, producing 104 matches across the group stage and knockout rounds.

Using machine learning, historical statistics, and soccer domain knowledge, predict match scores, corners, and cards for every fixture. You must submit all your predictions before a single ball is kicked.

The scoring system rewards precision: an exact scoreline earns maximum points, while close predictions still earn partial credit. Later rounds carry score multipliers, so a strong model that holds up in the knockout stages can leapfrog the competition. The challenge is designed to be difficult enough that no one can achieve a perfect score—even with AI assistance—but accessible enough that any data enthusiast can participate and score points.

## 💾 The data

You have access to the following files:

#### `data/group_fixtures.csv` — all 72 group stage matches
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `group` | Group letter (A–L) |
| `home_team` | Home team name |
| `away_team` | Away team name |
| `date` | Match date (UTC) |
| `venue` | Stadium and city |

#### `data/knockout_slots.csv` — all 32 knockout round slots
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `round` | Round name (e.g. `Quarter-final`) |
| `multiplier` | Score multiplier for this round |
| `slot_home` | Description of the home team slot (e.g. `Winner Group A`) |
| `slot_away` | Description of the away team slot |

| Variable | Description |
|---|---|

You may also bring in any external data—FIFA rankings, historical match results, player statistics—to build your predictions.

In [5]:
import pandas as pd

group_fixtures = pd.read_csv('data/group_fixtures.csv')
group_fixtures.head()

,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver"


In [6]:
knockout_slots = pd.read_csv('data/knockout_slots.csv')
knockout_slots

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F)
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H)
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I)
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K)
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J)
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J)


In [9]:
import numpy as np
from scipy.stats import poisson
 
np.random.seed(42)

In [10]:
# Current UEFA Ratings
TEAM_STRENGTH = {
    "Argentina": 98, "France": 97, "England": 96, "Brazil": 95,
    "Portugal": 94, "Spain": 93, "Germany": 92, "Netherlands": 91,
    "Belgium": 90, "Croatia": 89, "Uruguay": 88, "Colombia": 87,
    "Mexico": 86, "USA": 85, "Morocco": 84, "Senegal": 83,
    "Japan": 82, "South Korea": 81, "Australia": 80, "Canada": 79,
    "Ecuador": 78, "Chile": 77, "Paraguay": 76, "Peru": 75,
    "Venezuela": 72, "Bolivia": 70,
    "Nigeria": 82, "Ghana": 79, "Cameroon": 78, "Ivory Coast": 81,
    "Algeria": 80, "Tunisia": 79, "Egypt": 81, "South Africa": 74,
    "IR Iran": 77, "Saudi Arabia": 75, "Qatar": 72, "Iraq": 73,
    "Denmark": 88, "Switzerland": 87, "Austria": 85, "Poland": 83,
    "Czech Republic": 82, "Serbia": 83, "Hungary": 80, "Romania": 79,
    "Slovakia": 78, "Ukraine": 82, "Turkey": 81, "Sweden": 84,
    "Norway": 83, "Scotland": 80, "Wales": 79, "Ireland": 78,
    "Greece": 78, "Albania": 74, "Slovenia": 78, "Iceland": 77,
    "Finland": 75, "North Macedonia": 72,
    "New Zealand": 70, "Costa Rica": 75, "Honduras": 72,
    "Jamaica": 71, "El Salvador": 70, "Panama": 73,
    "Cuba": 65, "Haiti": 68,
    "India": 68, "China PR": 72, "Thailand": 69,
    "Indonesia": 67, "Malaysia": 66, "Uzbekistan": 74,
    # Playoff slots — assigned mid-range strength
    "UEFA Playoff A": 79, "UEFA Playoff B": 79,
    "UEFA Playoff C": 79, "UEFA Playoff D": 79,
    "CONMEBOL/CONCACAF": 76,
    "Inter-confederation playoff 1": 75,
    "Inter-confederation playoff 2": 75,
    "Best 3rd team": 76,
}

In [11]:
DEFAULT_STRENGTH = 75
 
def get_strength(team_name: str) -> int:
    """Look up team strength, with fuzzy fallback."""
    if not team_name:
        return DEFAULT_STRENGTH
    if team_name in TEAM_STRENGTH:
        return TEAM_STRENGTH[team_name]
    # Fuzzy match
    for k, v in TEAM_STRENGTH.items():
        if team_name.lower() in k.lower() or k.lower() in team_name.lower():
            return v
    return DEFAULT_STRENGTH

In [12]:
# Prediction Mechanism

BASE_GOALS_PER_GAME = 1.15   # league-average goals per team per game
HOME_ADVANTAGE      = 0.08   # +8% for home team lambda

 
def predict_match(home_team: str, away_team: str, is_knockout: bool = False) -> dict:
    """
    Predict a single match outcome.
 
    Returns a dict with:
        predicted_home_goals, predicted_away_goals,
        corners, yellow_cards, red_cards,
        winning_team / match_winner,
        penalties (knockout only)
    """
    hs = get_strength(home_team)
    as_ = get_strength(away_team)
 
    # Poisson lambdas — scaled by team strength relative to a baseline of 90
    home_lambda = BASE_GOALS_PER_GAME * (hs / 90) * (1 + HOME_ADVANTAGE)
    away_lambda = BASE_GOALS_PER_GAME * (as_ / 90)
 
    hg = poisson.rvs(home_lambda)
    ag = poisson.rvs(away_lambda)
 
    # Ancillary stats scaled by match intensity (avg of both strengths)
    intensity = (hs + as_) / 200          # normalised 0-1
    ko_bonus  = 1 if is_knockout else 0
 
    corners      = int(np.round(8 + intensity * 4 + ko_bonus))
    yellow_cards = int(np.round(2 + intensity * 2 + ko_bonus * 0.5))
    red_cards    = int(np.random.choice([0, 1], p=[0.92, 0.08]))
 
    penalties = False
    if hg > ag:
        winner = home_team
    elif ag > hg:
        winner = away_team
    else:
        if is_knockout:
            # Extra time → penalties (50/50 simplification)
            winner    = np.random.choice([home_team, away_team])
            penalties = True
        else:
            winner = "Draw"
 
    return {
        "predicted_home_goals": hg,
        "predicted_away_goals": ag,
        "corners":      corners,
        "yellow_cards": yellow_cards,
        "red_cards":    red_cards,
        "winner":       winner,
        "penalties":    penalties,
    }

## 💪 Competition challenge

The 2026 World Cup has two phases:

- **Group stage** (matches 1–72): The 48 teams are split into 12 groups of 4. Every team plays the other 3 teams in their group once. The best teams from each group advance to the next phase.
- **Knockout stage** (matches 73–104): Single-elimination rounds — Round of 32, Round of 16, Quarter-finals, Semi-finals, and the Final. Lose once and you're out. Crucially, the two teams playing in each knockout match are not known in advance: they depend on who qualified from the group stage.

Submit predictions for **every match** in both phases. For each match you need to predict:

1. **Score** — the exact final scoreline (e.g. `2-1` means the home team scores 2, the away team scores 1). For knockout matches, the score is the result after 90 minutes and extra time — the penalty shootout is not included.
2. **Corners** — the number of corner kicks awarded in the match
3. **Yellow cards** — the number of yellow cards shown in the match
4. **Red cards** — the number of red cards shown in the match

For **group stage** matches, also predict:
- **Winning team** — which team wins the individual match (use `home`, `away`, or `draw`)

For **knockout round** matches, also predict:
- **Matchup** — which two teams you predict will be playing in that slot. Because the bracket is determined by group stage results, you need to predict which teams advance far enough to meet in each round.
- **Match winner** — which team wins the match (use `home` or `away`)
- **Penalties** — whether the match goes to a penalty shootout (`True` or `False`)

### Scoring system

| Category | Condition | Points |
|---|---|---|
| Score | Exact scoreline | 25 |
| Score | Correct goal difference, wrong score | 10 |
| Score | Correct total goals, wrong score | 10 |
| Corners | Exact number | 10 |
| Corners | Off by 2 | 5 |
| Yellow cards | Exact number | 10 |
| Yellow cards | Off by 1 | 5 |
| Red cards | Exact number | 5 |
| Winning team *(group stage only)* | Correct | 40 |
| Matchup *(knockout only)* | Both teams correct | 20 |
| Matchup *(knockout only)* | One team correct | 10 |
| Match winner *(knockout only)* | Correct | 20 |
| Penalties *(knockout only)* | Correct | 5 |

All points for a match are multiplied by the round factor:

| Round | Multiplier |
|---|---|
| Group stage | ×1 |
| Round of 32 | ×1 |
| Round of 16 | ×2 |
| Quarter-final | ×4 |
| Semi-final | ×8 |
| Third-place playoff | ×8 |
| Final | ×16 |

## 🗓️ Group stage predictions

Fill in your predictions for all 72 group stage matches below.

In [13]:
group_predictions = group_fixtures.copy()

# Fill in your predictions for each match
# Example (match 1 — Mexico vs South Africa): predicted_home_goals=2, predicted_away_goals=1, corners=9, yellow_cards=3, red_cards=0, winning_team='home'
group_predictions['predicted_home_goals'] = None   # e.g. 2
group_predictions['predicted_away_goals'] = None   # e.g. 1
group_predictions['corners']              = None   # e.g. 9
group_predictions['yellow_cards']         = None   # e.g. 3
group_predictions['red_cards']            = None   # e.g. 0
group_predictions['winning_team']         = None   # "home", "away", or "draw"

#group_predictions

In [15]:

def predict_group_stage(df: pd.DataFrame) -> pd.DataFrame:
    results = []
    for _, row in df.iterrows():
        p = predict_match(row["home_team"], row["away_team"], is_knockout=False)
        results.append({
            **row.to_dict(),
            "predicted_home_goals": p["predicted_home_goals"],
            "predicted_away_goals": p["predicted_away_goals"],
            "corners":              p["corners"],
            "yellow_cards":         p["yellow_cards"],
            "red_cards":            p["red_cards"],
            "winning_team":         p["winner"],
        })
    col_order = [
        "match_id", "group", "home_team", "away_team", "date_utc", "venue",
        "predicted_home_goals", "predicted_away_goals",
        "corners", "yellow_cards", "red_cards", "winning_team",
    ]
    return pd.DataFrame(results)[col_order]
 

## 🏆 Knockout stage predictions

For knockout matches you also predict **which teams are playing**. Fill in the team names based on your group stage predictions, then add your match predictions.

In [14]:
knockout_predictions = knockout_slots.copy()

# Fill in your predictions for each knockout match
# Example (match 73 — Round of 32): predicted_home_team='Brazil', predicted_away_team='France', predicted_home_goals=1, predicted_away_goals=0, corners=8, yellow_cards=2, red_cards=0, match_winner='home', penalties=False
knockout_predictions['predicted_home_team']  = None   # e.g. "Brazil"
knockout_predictions['predicted_away_team']  = None   # e.g. "France"
knockout_predictions['predicted_home_goals'] = None   # e.g. 1
knockout_predictions['predicted_away_goals'] = None   # e.g. 0
knockout_predictions['corners']              = None   # e.g. 8
knockout_predictions['yellow_cards']         = None   # e.g. 2
knockout_predictions['red_cards']            = None   # e.g. 0
knockout_predictions['match_winner']         = None   # "home" or "away"
knockout_predictions['penalties']            = None   # True or False

#knockout_predictions

In [16]:

def build_group_standings(group_preds_df: pd.DataFrame) -> dict:
    """
    Derive simple group standings (by points, then goal diff) from predictions.
    Returns { group_letter: [team1_winner, team2_runner_up, ...] }
    """
    standings = {}
    groups = group_preds_df["group"].unique()
 
    for grp in groups:
        matches = group_preds_df[group_preds_df["group"] == grp]
        teams   = pd.unique(matches[["home_team", "away_team"]].values.ravel())
        stats   = {t: {"pts": 0, "gf": 0, "ga": 0} for t in teams}
 
        for _, m in matches.iterrows():
            ht, at = m["home_team"], m["away_team"]
            hg, ag = m["predicted_home_goals"], m["predicted_away_goals"]
            stats[ht]["gf"] += hg; stats[ht]["ga"] += ag
            stats[at]["gf"] += ag; stats[at]["ga"] += hg
            if hg > ag:
                stats[ht]["pts"] += 3
            elif ag > hg:
                stats[at]["pts"] += 3
            else:
                stats[ht]["pts"] += 1
                stats[at]["pts"] += 1
 
        ranked = sorted(
            teams,
            key=lambda t: (stats[t]["pts"], stats[t]["gf"] - stats[t]["ga"], stats[t]["gf"]),
            reverse=True,
        )
        standings[grp] = ranked
 
    return standings
 

In [22]:
_slot_cache: dict = {}
 
def resolve_slot(slot: str, standings: dict, ko_df: pd.DataFrame) -> str:
    """Resolve a positional slot string to an actual team name."""
    if slot in _slot_cache:
        return _slot_cache[slot]
 
    team = _resolve_slot_inner(slot, standings, ko_df)
    _slot_cache[slot] = team
    return team
 
 
def _resolve_slot_inner(slot: str, standings: dict, ko_df: pd.DataFrame) -> str:
    if slot.startswith("Winner Group "):
        grp = slot.replace("Winner Group ", "")
        return standings.get(grp, [None])[0] or slot
 
    if slot.startswith("Runner-up Group "):
        grp = slot.replace("Runner-up Group ", "")
        ranked = standings.get(grp, [])
        return ranked[1] if len(ranked) > 1 else (ranked[0] if ranked else slot)
 
    if slot.startswith("Best 3rd"):
        # Collect all group 3rd-place teams, return the strongest
        candidates = []
        for grp, ranked in standings.items():
            if len(ranked) >= 3:
                candidates.append(ranked[2])
        if candidates:
            return max(candidates, key=get_strength)
        return "Best 3rd team"
 
    if slot.startswith("Winner Match "):
        mid = int(slot.replace("Winner Match ", ""))
        row = ko_df[ko_df["match_id"] == mid]
        if row.empty:
            return slot
        r = row.iloc[0]
        ht = resolve_slot(r["slot_home"], standings, ko_df)
        at = resolve_slot(r["slot_away"], standings, ko_df)
        return ht if get_strength(ht) >= get_strength(at) else at
 
    if slot.startswith("Loser Match "):
        mid = int(slot.replace("Loser Match ", ""))
        row = ko_df[ko_df["match_id"] == mid]
        if row.empty:
            return slot
        r = row.iloc[0]
        ht = resolve_slot(r["slot_home"], standings, ko_df)
        at = resolve_slot(r["slot_away"], standings, ko_df)
        return at if get_strength(ht) >= get_strength(at) else ht
 
    return slot
 
# KNOCKOUT STAGE PREDICTIONS

def predict_knockout_stage(
    ko_df: pd.DataFrame,
    standings: dict,
) -> pd.DataFrame:
    global _slot_cache
    _slot_cache = {}   # reset cache for clean run
 
    results = []
    # Process in match_id order so slot resolution cascades correctly
    for _, row in ko_df.sort_values("match_id").iterrows():
        ht = resolve_slot(row["slot_home"], standings, ko_df)
        at = resolve_slot(row["slot_away"], standings, ko_df)
        p  = predict_match(ht, at, is_knockout=True)
 
        results.append({
            "match_id":              row["match_id"],
            "round":                 row["round"],
            "multiplier":            row["multiplier"],
            "date_utc":              row["date_utc"],
            "venue":                 row["venue"],
            "slot_home":             row["slot_home"],
            "slot_away":             row["slot_away"],
            "predicted_home_team":   ht,
            "predicted_away_team":   at,
            "predicted_home_goals":  p["predicted_home_goals"],
            "predicted_away_goals":  p["predicted_away_goals"],
            "corners":               p["corners"],
            "yellow_cards":          p["yellow_cards"],
            "red_cards":             p["red_cards"],
            "match_winner":          p["winner"],
            "penalties":             p["penalties"],
        })
 
    col_order = [
        "match_id", "round", "multiplier", "date_utc", "venue",
        "slot_home", "slot_away",
        "predicted_home_team", "predicted_away_team",
        "predicted_home_goals", "predicted_away_goals",
        "corners", "yellow_cards", "red_cards",
        "match_winner", "penalties",
    ]
    return pd.DataFrame(results)[col_order]
 
 
# =============================================================================
# 7. RUN THE MODEL
# =============================================================================
 
group_predictions   = predict_group_stage(group_fixtures)
standings              = build_group_standings(group_predictions)
knockout_predictions_df = predict_knockout_stage(knockout_slots, standings)


In [23]:
group_predictions_df

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",2,1,11,4,0,Mexico
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",0,3,11,4,1,UEFA Playoff D
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto",1,0,11,4,0,Canada
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",0,1,11,4,0,Paraguay
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver",1,0,11,4,0,Australia
...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia",1,1,11,4,0,Draw
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami",2,1,12,4,0,Colombia
69,70,K,FIFA Playoff 1,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta",1,2,11,3,0,Uzbekistan
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",1,2,11,4,1,Austria


In [24]:
knockout_predictions_df

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,Mexico,Canada,1,1,12,4,0,Mexico,True
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,Scotland,Japan,1,0,12,4,0,Scotland,False
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),Curaçao,England,1,1,12,4,0,Curaçao,True
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,Tunisia,Morocco,1,0,12,4,0,Tunisia,False
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,Germany,Senegal,3,1,12,4,0,Germany,False
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),France,England,1,2,13,4,0,England,False
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),South Korea,England,3,0,13,4,0,South Korea,False
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),Ghana,England,1,2,12,4,0,England,False
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),Belgium,England,2,0,13,4,1,Belgium,False
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),Australia,England,0,2,13,4,0,England,False


## ✅ Checklist before publishing into the competition

- Rename your workspace to make it descriptive of your work. N.B. you should leave the notebook name as `notebook.ipynb`.
- Remove redundant cells like the judging criteria, so the workbook is focused on your predictions.
- Make sure all prediction cells are filled in—`None` values will score 0 points.
- Check that all cells run without error.
- Make sure your workbook is published before **June 10, 2026 at 09:00 UTC**.

## ⏳ Time is ticking. Good luck!